# Milwaukee reliever appearances

Explore the pitch-level Statcast source and derive one row per Milwaukee pitcher appearance. The transformation remains in this notebook while its assumptions are investigated and validated.

## 1. Setup

Use the project's configured DuckDB connection so Azure authentication and extensions stay in one place.

In [ ]:
%load_ext autoreload
%autoreload 2

from crooked_numbers_analytics.db import open_database
from crooked_numbers_analytics.paths import azure_dataset_path

con = open_database()

## 2. Research parameters

Keep the initial comparison period and team in one place so later queries do not embed them.

In [ ]:
seasons = [2023, 2024]
team = "MIL"

## 3. Source paths

Build one narrow season-level glob per selected season. This avoids a recursive scan of unrelated Statcast partitions.

In [ ]:
season_paths = {
    season: azure_dataset_path(
        f"raw/statcast/season={season}/game_date=*/statcast.parquet"
    )
    for season in seasons
}

quoted_paths = ", ".join(
    f"'{path.replace(chr(39), chr(39) * 2)}'"
    for path in season_paths.values()
)
con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW statcast_source AS
    SELECT *
    FROM read_parquet(
        [{quoted_paths}],
        hive_partitioning = true,
        union_by_name = true
    )
    """
)
season_paths

## 4. Inspect the source data

First inspect metadata and small query results. DuckDB reads the Parquet files directly; only these limited results become pandas DataFrames for display.

In [ ]:
con.sql("DESCRIBE statcast_source").df()

In [ ]:
con.sql(
    """
    SELECT
        season, game_date, game_pk, game_type, inning, inning_topbot,
        at_bat_number, pitch_number, home_team, away_team,
        pitcher, player_name, batter, events, description
    FROM statcast_source
    ORDER BY game_date, game_pk, at_bat_number, pitch_number
    LIMIT 10
    """
).df()

In [ ]:
con.sql(
    """
    SELECT
        game_type, COUNT(*) AS pitch_rows,
        COUNT(DISTINCT game_pk) AS games
    FROM statcast_source
    GROUP BY game_type
    ORDER BY game_type
    """
).df()

In [ ]:
con.execute(
    """
    SELECT home_team, away_team, COUNT(DISTINCT game_pk) AS games
    FROM statcast_source
    WHERE home_team = ? OR away_team = ?
    GROUP BY home_team, away_team
    ORDER BY games DESC, home_team, away_team
    LIMIT 20
    """,
    [team, team],
).df()

The source has `home_team` and `away_team`, but no direct pitching-team field. In the top half, the home team pitches; in the bottom half, the away team pitches. The pitch sequence fields are `at_bat_number` and `pitch_number`. The next sample exposes those fields alongside pitcher and batter IDs so their meanings can be checked directly.

In [ ]:
con.execute(
    """
    SELECT
        game_date, game_pk, inning, inning_topbot,
        at_bat_number, pitch_number, pitcher, player_name, batter, des
    FROM statcast_source
    WHERE game_type = 'R'
      AND (home_team = ? OR away_team = ?)
    ORDER BY game_date, game_pk, at_bat_number, pitch_number
    LIMIT 20
    """,
    [team, team],
).df()

## 5. Regular-season Milwaukee pitching scope

Materialize only the selected team's regular-season pitching rows in DuckDB. The `CASE` expression implements the half-inning rule above. Keeping this small working table avoids repeatedly reading the remote season data during appearance exploration.

In [ ]:
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE team_regular_season_pitches AS
    SELECT
        season, game_pk, game_date, game_type, home_team, away_team,
        inning, inning_topbot, at_bat_number, pitch_number, pitch_uid,
        pitcher, player_name, batter, des,
        CASE
            WHEN inning_topbot = 'Top' THEN home_team
            WHEN inning_topbot = 'Bot' THEN away_team
        END AS pitching_team
    FROM statcast_source
    WHERE game_type = 'R'
      AND CASE
              WHEN inning_topbot = 'Top' THEN home_team
              WHEN inning_topbot = 'Bot' THEN away_team
          END = ?
    """,
    [team],
)

con.sql(
    """
    SELECT
        season, pitching_team, COUNT(*) AS pitch_rows,
        COUNT(DISTINCT game_pk) AS games
    FROM team_regular_season_pitches
    GROUP BY season, pitching_team
    ORDER BY season
    """
).df()

`player_name` is not accepted as the pitcher name merely from its label. This checks whether it maps consistently to `pitcher` while each pitcher faces multiple batter IDs. Any row returned by the second query needs investigation before using the name field.

In [ ]:
con.sql(
    """
    SELECT
        pitcher, MIN(player_name) AS player_name,
        COUNT(DISTINCT batter) AS batters_faced, COUNT(*) AS pitch_rows
    FROM team_regular_season_pitches
    GROUP BY pitcher
    ORDER BY pitch_rows DESC
    LIMIT 15
    """
).df()

In [ ]:
con.sql(
    """
    SELECT
        pitcher, COUNT(DISTINCT player_name) AS name_count,
        LIST(DISTINCT player_name ORDER BY player_name) AS names
    FROM team_regular_season_pitches
    GROUP BY pitcher
    HAVING COUNT(DISTINCT player_name) <> 1
    ORDER BY pitcher
    """
).df()

## 6. Derive pitcher appearances

Assign a within-team/game pitch sequence from `at_bat_number` and `pitch_number`, then aggregate by game, pitching team, and pitcher. The verified `player_name` value is exposed as `pitcher_name`. `pitch_count` counts pitch-level source rows.

In [ ]:
con.execute(
    """
    CREATE OR REPLACE TEMP VIEW pitcher_appearances AS
    WITH ordered_pitches AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY game_pk, pitching_team
                ORDER BY at_bat_number, pitch_number, pitcher
            ) AS game_pitch_order
        FROM team_regular_season_pitches
    ),
    appearances AS (
        SELECT
            season, game_pk, MIN(game_date) AS game_date, pitching_team, pitcher,
            ARG_MIN(player_name, game_pitch_order) AS pitcher_name,
            COUNT(*) AS pitch_count,
            MIN(inning) AS first_inning,
            MAX(inning) AS last_inning,
            MIN(game_pitch_order) AS first_pitch_order
        FROM ordered_pitches
        GROUP BY season, game_pk, pitching_team, pitcher
    ),
    ranked_appearances AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY game_pk, pitching_team
                ORDER BY first_pitch_order, pitcher
            ) AS appearance_order
        FROM appearances
    )
    SELECT
        season, game_pk, game_date, pitching_team, pitcher, pitcher_name,
        pitch_count, first_inning, last_inning, appearance_order,
        appearance_order = 1 AS is_starter
    FROM ranked_appearances
    """
)

con.sql(
    """
    SELECT *
    FROM pitcher_appearances
    ORDER BY game_date, game_pk, appearance_order
    LIMIT 25
    """
).df()

## 7. Starter vs. reliever

The starter is the first pitcher to throw a recorded pitch for the team, not merely any pitcher recorded in inning one. With one appearance row per pitcher, the earliest `first_pitch_order` receives `appearance_order = 1`; all other appearances are relief appearances.

In [ ]:
con.execute(
    """
    CREATE OR REPLACE TEMP VIEW reliever_appearances AS
    SELECT *
    FROM pitcher_appearances
    WHERE NOT is_starter
    """
)

con.sql(
    """
    SELECT *
    FROM reliever_appearances
    ORDER BY game_date, game_pk, appearance_order
    LIMIT 25
    """
).df()

## 8. Validate individual games

Select a few games deterministically and list their pitchers in appearance order. These compact tables are intended for manual comparison with MLB box scores when needed.

In [ ]:
con.execute(
    """
    CREATE OR REPLACE TEMP VIEW validation_games AS
    SELECT DISTINCT season, game_date, game_pk
    FROM pitcher_appearances
    ORDER BY game_date, game_pk
    LIMIT 3
    """
)

con.sql(
    """
    SELECT
        a.game_date, a.game_pk, a.pitcher_name, a.is_starter,
        a.pitch_count, a.first_inning, a.last_inning, a.appearance_order
    FROM pitcher_appearances AS a
    INNER JOIN validation_games AS g
        USING (season, game_date, game_pk)
    ORDER BY a.game_date, a.game_pk, a.appearance_order
    """
).df()

## 9. Basic sanity checks

Summarize coverage and pitch-count ranges, then explicitly check the expected uniqueness and one-starter-per-game invariants. Empty duplicate results and zero anomalous games are the desired outcomes.

In [ ]:
con.sql(
    """
    SELECT
        season, COUNT(DISTINCT game_pk) AS regular_season_games,
        COUNT(*) AS pitcher_appearances,
        COUNT(*) FILTER (WHERE NOT is_starter) AS relief_appearances,
        MIN(pitch_count) AS min_pitch_count,
        MAX(pitch_count) AS max_pitch_count
    FROM pitcher_appearances
    GROUP BY season
    ORDER BY season
    """
).df()

In [ ]:
con.sql(
    """
    SELECT game_pk, pitching_team, pitcher, COUNT(*) AS appearance_rows
    FROM pitcher_appearances
    GROUP BY game_pk, pitching_team, pitcher
    HAVING COUNT(*) > 1
    ORDER BY game_pk, pitcher
    """
).df()

In [ ]:
con.sql(
    """
    WITH starters_by_game AS (
        SELECT
            season, game_pk, pitching_team,
            COUNT(*) FILTER (WHERE is_starter) AS starter_count
        FROM pitcher_appearances
        GROUP BY season, game_pk, pitching_team
    )
    SELECT
        COUNT(*) FILTER (WHERE starter_count = 0) AS games_with_no_starter,
        COUNT(*) FILTER (WHERE starter_count > 1) AS games_with_multiple_starters,
        MIN(starter_count) AS min_starters,
        MAX(starter_count) AS max_starters
    FROM starters_by_game
    """
).df()